[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00


In [2]:
import torch
import math

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation

    B, S, D = q.shape

    thetas = 2 * torch.arange(D // 2) / D
    thetas = torch.pow(10000, -thetas).unsqueeze(0)

    m = torch.arange(S).unsqueeze(1)

    m_thetas = thetas * m
    m_thetas = torch.cat([m_thetas, m_thetas], dim=-1)

    cos = torch.cos(m_thetas)
    sin = torch.sin(m_thetas)

    def rotate_half(x):
      x1 = x[..., : x.shape[-1] // 2]
      x2 = x[..., x.shape[-1] // 2 : ]
      return torch.cat([-x2, x1], dim=-1)

    q_rope = q * cos + rotate_half(q) * sin
    k_rope = k * cos + rotate_half(k) * sin
    return q_rope, k_rope


In [8]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (4.4ms)
  ✅ [2/4] Preserves norm (5.2ms)
  ✅ [3/4] Relative position property (10.6ms)
  ✅ [4/4] Gradient flow (18.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.6ms total)
  Progress saved. Run status() to see your dashboard.

